# Sieci neuronowe i deep learning - Lab 5

## CNN i transfer learning

PyTorch.

Pracujemy na **CIFAR-10**. Celem jest porownanie CNN trenowanego od zera z modelem wykorzystujacym backbone pretrained na ImageNet.

Do punktacji bazowym zbiorem jest CIFAR-10. Wlasny/trudniejszy zbior moze byc rozszerzeniem, ale musi zachowac zgodny preprocessing, format etykiet i liczbe klas w ostatniej warstwie.

## Zadanie 1

- Zaladuj CIFAR-10 i pokaz przykladowe obrazy.
- Opcjonalna podmiana datasetu nie zastepuje punktowanego wariantu bazowego bez zgody prowadzacego.
- Zbuduj CNN trenowany od zera.
- Wytrenuj model i narysuj krzywe uczenia.
- Porownaj accuracy treningowe i walidacyjne.

In [ ]:
# Konfiguracja techniczna.
# Ta komorka jest gotowa, bo dotyczy glownie cache danych i importow.

import os
from pathlib import Path


def configure_torch_cache():
    candidates = [
        Path("../dane/torchvision"),
        Path("laboratoria/dane/torchvision"),
        Path("dane/torchvision"),
        Path("/content/dane/torchvision"),
    ]
    for candidate in candidates:
        if candidate.parent.exists():
            candidate.mkdir(parents=True, exist_ok=True)
            model_cache = candidate / "models"
            model_cache.mkdir(parents=True, exist_ok=True)
            os.environ["TORCH_HOME"] = str(model_cache.resolve())
            return candidate.resolve(), model_cache.resolve()
    fallback = Path("dane/torchvision")
    fallback.mkdir(parents=True, exist_ok=True)
    model_cache = fallback / "models"
    model_cache.mkdir(parents=True, exist_ok=True)
    os.environ["TORCH_HOME"] = str(model_cache.resolve())
    return fallback.resolve(), model_cache.resolve()


DATA_ROOT, TORCH_HOME = configure_torch_cache()

import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, models, transforms
from torchvision.models import MobileNet_V2_Weights

from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split


RANDOM_STATE = 42
BATCH_SIZE = 128
IMAGE_SIZE = 96
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(RANDOM_STATE)

CLASS_NAMES = ["airplane", "automobile", "bird", "cat", "deer", "dog", "frog", "horse", "ship", "truck"]
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]


In [ ]:
# Zadanie 1a: dane i transformacje.

# TODO:
# - pobierz CIFAR-10,
# - wydziel train/validation ze zbioru treningowego,
# - przygotuj osobne transformacje dla baseline, modelu regularyzowanego i transfer learning.

# Zbiór treningowy CIFAR-10: 50 000 obrazów 32x32 RGB, 10 klas po 5000 obrazów.
# Wczytujemy go tu bez transformacji - potrzebne są tylko etykiety do podziału i surowe piksele.
base_train = datasets.CIFAR10(root=DATA_ROOT, train=True, download=True)

all_indices = np.arange(len(base_train))
# 10% (5000 obrazów) idzie na walidację; stratify -> w walidacji jest dokładnie po 500 obrazów z każdej klasy.
# Zbiór testowy (osobne 10 000 obrazów) ruszamy dopiero w wyniku końcowym.
train_indices, val_indices = train_test_split(
    all_indices,
    test_size=0.1,
    stratify=base_train.targets,
    random_state=RANDOM_STATE,
)

# Średnia i odchylenie każdego kanału RGB liczone tylko na części treningowej (bez podglądania walidacji).
# base_train.data ma kształt (50000, 32, 32, 3) i typ uint8.
train_pixels = base_train.data[train_indices].astype("float32") / 255.0
CIFAR_MEAN = train_pixels.mean(axis=(0, 1, 2)).tolist()
CIFAR_STD = train_pixels.std(axis=(0, 1, 2)).tolist()
del train_pixels

# Baseline: tylko zamiana na tensor i normalizacja, bez żadnej augmentacji.
scratch_train_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
])
scratch_eval_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
])

# Model regularyzowany: losowe przesunięcie (wycinek 32x32 z obrazu dopełnionego 4 pikselami z każdej strony)
# i losowe odbicie w poziomie. Obie zmiany nie zmieniają klasy obiektu (samolot odbity dalej jest samolotem).
# Walidacja używa scratch_eval_transform - bez augmentacji.
regularized_train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
])

# Transfer learning: MobileNetV2 uczył się na ImageNet (obrazy ok. 224x224), więc powiększamy CIFAR do 96x96
# i normalizujemy statystykami ImageNet - takimi samymi, jakie widział backbone podczas treningu.
transfer_train_transform = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
transfer_eval_transform = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

NUM_WORKERS = 4  # procesy wczytujące dane równolegle z treningiem na GPU


def make_loaders(train_transform, val_transform, batch_size=BATCH_SIZE):
    # TODO: utworz DataLoadery dla train i validation, uzywajac Subset.
    # Dwie instancje CIFAR10 różnią się tylko transformacją; Subset wybiera z nich nasze indeksy.
    train_dataset = Subset(datasets.CIFAR10(root=DATA_ROOT, train=True, transform=train_transform), train_indices)
    val_dataset = Subset(datasets.CIFAR10(root=DATA_ROOT, train=True, transform=val_transform), val_indices)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,
                              num_workers=NUM_WORKERS, pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False,
                            num_workers=NUM_WORKERS, pin_memory=True)
    return train_loader, val_loader


print("urządzenie:", DEVICE)
print("train:", len(train_indices), "| validation:", len(val_indices))
print("obrazów na klasę w train:", np.bincount(np.array(base_train.targets)[train_indices]))
print("średnia RGB (train):", np.round(CIFAR_MEAN, 3), "| odchylenie RGB:", np.round(CIFAR_STD, 3))

In [ ]:
# Zadanie 1b: podglad danych.

# TODO: pokaz siatke przykladowych obrazow z podpisami klas.
# Warto sprawdzic, czy etykiety zgadzaja sie z obrazami.

# base_train nie ma transformacji, więc Subset zwraca pary (obraz PIL, etykieta) - od razu do wyświetlenia.
preview_dataset = Subset(base_train, train_indices)
fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for ax, idx in zip(axes.ravel(), range(16)):
    image, label = preview_dataset[idx]
    ax.imshow(image)
    ax.set_title(CLASS_NAMES[label], fontsize=9)
    ax.axis("off")
plt.suptitle("CIFAR-10: przykładowe obrazy treningowe (32x32)")
plt.tight_layout()
plt.show()

In [ ]:
# Funkcje pomocnicze.
# Tu nie powinno byc logiki modelu, tylko wspolna petla treningowa i narzedzia do porownan.

import copy
import time


def plot_history(history, title):
    # TODO: narysuj loss i accuracy dla train/validation.
    epochs = range(1, len(history["loss"]) + 1)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].plot(epochs, history["loss"], label="train")
    axes[0].plot(epochs, history["val_loss"], label="validation")
    axes[0].set_title(f"{title} - loss")
    axes[0].set_xlabel("epoka")
    axes[0].grid(True)
    axes[0].legend()

    axes[1].plot(epochs, history["accuracy"], label="train")
    axes[1].plot(epochs, history["val_accuracy"], label="validation")
    axes[1].set_title(f"{title} - accuracy")
    axes[1].set_xlabel("epoka")
    axes[1].grid(True)
    axes[1].legend()

    plt.tight_layout()
    plt.show()


def print_score_table(rows):
    # TODO: wypisz porownanie modeli w czytelnej tabeli.
    header = (f"{'model':<34} {'tren. parametry':>15} {'train_acc':>10} {'val_acc':>9} "
              f"{'różnica':>8} {'epoka':>6} {'czas [s]':>9}")
    print(header)
    print("-" * len(header))
    for r in rows:
        gap = r["train_accuracy"] - r["val_accuracy"]
        print(f"{r['name']:<34} {r['params']:>15,d} {r['train_accuracy']:>10.4f} {r['val_accuracy']:>9.4f} "
              f"{gap:>8.4f} {r['best_epoch']:>6d} {r['time_s']:>9.0f}")


def count_parameters(model):
    # TODO: policz tylko trenowalne parametry.
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def keep_frozen_batchnorm_in_eval(model):
    # model.train() przełącza też BatchNorm w tryb treningowy, a wtedy BN nadpisuje swoje statystyki
    # (running_mean / running_var) statystykami z CIFAR - nawet gdy jego wagi są zamrożone.
    # Zamrożone warstwy BN (bez trenowalnych parametrów) zostawiamy więc w trybie eval,
    # żeby backbone działał dokładnie tak, jak po treningu na ImageNet.
    for module in model.modules():
        if isinstance(module, nn.BatchNorm2d) and not any(p.requires_grad for p in module.parameters()):
            module.eval()


def run_epoch(model, loader, loss_fn, optimizer=None):
    # TODO:
    # - obsluz tryb train/eval,
    # - przenies dane na DEVICE,
    # - policz loss i accuracy,
    # - wykonaj backward/step tylko w treningu.
    is_train = optimizer is not None
    model.train(is_train)  # włącza/wyłącza dropout i tryb BatchNorm
    if is_train:
        keep_frozen_batchnorm_in_eval(model)

    total_loss = 0.0
    correct = 0
    total = 0
    for X_batch, y_batch in loader:
        X_batch = X_batch.to(DEVICE, non_blocking=True)
        y_batch = y_batch.to(DEVICE, non_blocking=True)

        # W ewaluacji nie budujemy grafu obliczeń - szybciej i mniej pamięci.
        with torch.set_grad_enabled(is_train):
            logits = model(X_batch)
            loss = loss_fn(logits, y_batch)

        if is_train:
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        # loss.item() to średnia w batchu, więc mnożymy przez liczbę próbek, żeby dostać średnią z epoki.
        total_loss += loss.item() * len(y_batch)
        correct += (logits.argmax(dim=1) == y_batch).sum().item()
        total += len(y_batch)

    return total_loss / total, correct / total


def train_torch_model(model, name, train_loader, val_loader, epochs=10, learning_rate=0.001, weight_decay=0.0,
                      test_transform=None):
    # TODO: pelna petla treningowa z historia metryk i wierszem wyniku.
    model = model.to(DEVICE)
    loss_fn = nn.CrossEntropyLoss()
    # Optymalizator dostaje tylko parametry z requires_grad=True (ważne przy zamrożonym backbonie).
    trainable_params = [p for p in model.parameters() if p.requires_grad]
    # AdamW = Adam z weight decay odłączonym od gradientu; przy weight_decay=0 działa jak zwykły Adam.
    optimizer = torch.optim.AdamW(trainable_params, lr=learning_rate, weight_decay=weight_decay)

    history = {"loss": [], "val_loss": [], "accuracy": [], "val_accuracy": []}
    best_val_acc, best_epoch, best_state = -1.0, 0, None
    start = time.time()

    for epoch in range(1, epochs + 1):
        train_loss, train_acc = run_epoch(model, train_loader, loss_fn, optimizer)
        val_loss, val_acc = run_epoch(model, val_loader, loss_fn)

        history["loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["accuracy"].append(train_acc)
        history["val_accuracy"].append(val_acc)

        # Zapamiętujemy wagi z najlepszej epoki na walidacji (checkpoint zamiast early stoppingu).
        if val_acc > best_val_acc:
            best_val_acc, best_epoch = val_acc, epoch
            best_state = copy.deepcopy(model.state_dict())

        print(f"{name} | epoka {epoch:2d}/{epochs} | loss train {train_loss:.3f} val {val_loss:.3f} "
              f"| acc train {train_acc:.3f} val {val_acc:.3f}")

    elapsed = time.time() - start
    model.load_state_dict(best_state)  # wracamy do najlepszej epoki

    row = {
        "name": name,
        "params": count_parameters(model),
        "train_accuracy": history["accuracy"][best_epoch - 1],
        "val_accuracy": best_val_acc,
        "best_epoch": best_epoch,
        "time_s": elapsed,
        # Kopia modelu, bo ten sam obiekt może być dalej trenowany (fine-tuning po zamrożonym backbonie).
        "model_object": copy.deepcopy(model),
        "test_transform": test_transform,
    }
    return model, history, row

In [ ]:
# Zadanie 1c: CNN od zera.


class ScratchCNN(nn.Module):
    def __init__(self, dropout_rate=0.0):
        super().__init__()
        # TODO:
        # - zaprojektuj kilka blokow Conv2d,
        # - dodaj normalizacje/aktywacje/pooling,
        # - dodaj klasyfikator dla 10 klas,
        # - dropout_rate ma miec realny efekt.

        # Trzy bloki Conv 3x3 -> BatchNorm -> ReLU; po każdym bloku MaxPool 2x2 zmniejsza obraz o połowę.
        # Kanały: 3 -> 32 -> 64 -> 128, rozmiar: 32x32 -> 16x16 -> 8x8 -> 4x4.
        self.block1 = self._conv_block(3, 32)
        self.block2 = self._conv_block(32, 64)
        self.block3 = self._conv_block(64, 128)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

        # Dropout przed obiema warstwami Linear - tam jest prawie cała liczba parametrów modelu.
        # Przy dropout_rate=0.0 warstwy Dropout nic nie robią (baseline).
        self.classifier = nn.Sequential(
            nn.Flatten(),                 # (N, 128, 4, 4) -> (N, 2048)
            nn.Dropout(dropout_rate),
            nn.Linear(128 * 4 * 4, 256),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(256, 10),           # 10 logitów - po jednym na klasę
        )

    @staticmethod
    def _conv_block(in_channels, out_channels):
        # padding=1 przy filtrze 3x3 zachowuje wysokość i szerokość obrazu.
        return nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(),
        )

    def forward_features(self, x):
        # TODO: zwroc aktywacje z kilku warstw, zeby dac sie potem zwizualizowac.
        act1 = self.block1(x)                # (N, 32, 32, 32)
        act2 = self.block2(self.pool(act1))  # (N, 64, 16, 16)
        act3 = self.block3(self.pool(act2))  # (N, 128, 8, 8)
        return [act1, act2, act3]

    def forward(self, x):
        # TODO: forward klasyfikatora.
        act3 = self.forward_features(x)[-1]
        return self.classifier(self.pool(act3))  # (N, 128, 4, 4) -> (N, 10)


torch_rows = []
scratch_train_loader, scratch_val_loader = make_loaders(scratch_train_transform, scratch_eval_transform)

# TODO: wytrenuj baseline CNN od zera i dodaj jego wynik do torch_rows.
# Baseline: bez dropoutu, bez augmentacji, bez weight decay - punkt odniesienia dla Zadania 3.
SCRATCH_EPOCHS = 15
scratch_model = ScratchCNN(dropout_rate=0.0)
print("parametry ScratchCNN:", f"{count_parameters(scratch_model):,d}")

scratch_model, scratch_history, scratch_row = train_torch_model(
    scratch_model,
    "CNN od zera (baseline)",
    scratch_train_loader,
    scratch_val_loader,
    epochs=SCRATCH_EPOCHS,
    learning_rate=0.001,
    test_transform=scratch_eval_transform,
)
torch_rows.append(scratch_row)

plot_history(scratch_history, "CNN od zera (baseline)")
print_score_table(torch_rows)

## Zadanie 2

- Wyswietl filtry pierwszej warstwy konwolucyjnej modelu bazowego.
- Wyswietl mapy aktywacji wybranej warstwy konwolucyjnej dla jednego obrazu testowego.
- Krotko opisz, co widac w aktywacjach.

In [ ]:
# Zadanie 2: filtry i aktywacje.

# TODO:
# - wybierz pierwsza warstwe Conv2d,
# - wyswietl kilka filtrow,
# - pobierz jedna probke walidacyjna,
# - uzyj forward_features i wyswietl aktywacje wybranej warstwy.

conv_layer = scratch_model.block1[0]                # pierwsza warstwa Conv2d baseline'u
conv1_weights = conv_layer.weight.detach().cpu()    # (32, 3, 3, 3): 32 filtry, każdy 3x3 na 3 kanałach RGB
print("wagi pierwszej warstwy:", tuple(conv1_weights.shape))

# Każdy filtr skalujemy osobno do [0, 1], żeby dało się go narysować jako mały obraz RGB 3x3.
fig, axes = plt.subplots(4, 8, figsize=(10, 5.5))
for i, ax in enumerate(axes.ravel()):
    w = conv1_weights[i]
    w = (w - w.min()) / (w.max() - w.min() + 1e-8)
    ax.imshow(w.permute(1, 2, 0).numpy())            # (3, 3, 3) C,H,W -> H,W,C
    ax.set_title(f"filtr {i}", fontsize=8)
    ax.axis("off")
plt.suptitle("Filtry pierwszej warstwy Conv2d (3x3, RGB)")
plt.tight_layout()
plt.show()

# Jedna próbka walidacyjna (już po normalizacji z scratch_eval_transform).
SAMPLE_IDX = 0
sample_batch = scratch_val_loader.dataset[SAMPLE_IDX]
sample_image, sample_label = sample_batch
sample_input = sample_image.unsqueeze(0).to(DEVICE)  # (3, 32, 32) -> (1, 3, 32, 32)

scratch_model.eval()
with torch.no_grad():
    activations = scratch_model.forward_features(sample_input)
    sample_pred = scratch_model(sample_input).argmax(dim=1).item()

print("klasa prawdziwa:", CLASS_NAMES[sample_label], "| predykcja baseline:", CLASS_NAMES[sample_pred])
print("kształty aktywacji:", [tuple(a.shape) for a in activations])

# Oryginalny obraz (bez normalizacji) do porównania.
plt.figure(figsize=(2.5, 2.5))
plt.imshow(base_train.data[val_indices[SAMPLE_IDX]])
plt.title(f"obraz: {CLASS_NAMES[sample_label]}")
plt.axis("off")
plt.show()

# Po 16 pierwszych kanałów z bloku 1 (32x32) i bloku 2 (16x16). Jasne piksele = silna odpowiedź filtra.
for layer_name, act in [("blok 1 (32x32)", activations[0]), ("blok 2 (16x16)", activations[1])]:
    maps = act[0].cpu()  # (C, H, W)
    fig, axes = plt.subplots(2, 8, figsize=(12, 3.4))
    for c, ax in enumerate(axes.ravel()):
        ax.imshow(maps[c], cmap="viridis")
        ax.set_title(f"kanał {c}", fontsize=8)
        ax.axis("off")
    plt.suptitle(f"Mapy aktywacji po ReLU - {layer_name}")
    plt.tight_layout()
    plt.show()

## Zadanie 3

- Zastosuj metody poprawy generalizacji, np. augmentacje, dropout, L2/weight decay, batch normalization.
- Wytrenuj nowy model.
- Porownaj wynik walidacyjny z modelem bazowym.

In [ ]:
# Zadanie 3: generalizacja.

# TODO:
# - zbuduj drugi model CNN,
# - zastosuj co najmniej dwie techniki poprawy generalizacji,
# - uzasadnij wybrane wartosci dropout/weight_decay/augmentacji,
# - porownaj wynik z baseline.

# Ta sama architektura co baseline (BatchNorm jest w obu), żeby różnica wynikała tylko z regularyzacji:
# 1) augmentacja: RandomCrop(32, padding=4) + RandomHorizontalFlip (regularized_train_transform),
# 2) dropout 0.3 przed warstwami Linear,
# 3) weight decay 0.05 w AdamW.
# Model widzi co epokę trochę inne obrazy, więc uczy się dłużej - dajemy 30 epok zamiast 15.
REG_DROPOUT = 0.3
REG_WEIGHT_DECAY = 0.05
REG_EPOCHS = 30

regularized_train_loader, regularized_val_loader = make_loaders(regularized_train_transform, scratch_eval_transform)
regularized_model = ScratchCNN(dropout_rate=REG_DROPOUT)
regularized_model, regularized_history, regularized_row = train_torch_model(
    regularized_model,
    "CNN od zera + regularyzacja",
    regularized_train_loader,
    regularized_val_loader,
    epochs=REG_EPOCHS,
    learning_rate=0.001,
    weight_decay=REG_WEIGHT_DECAY,
    test_transform=scratch_eval_transform,
)

torch_rows.append(regularized_row)
plot_history(regularized_history, "CNN od zera + regularyzacja")
print_score_table(torch_rows)

## Zadanie 4

- Uzyj modelu pretrained na ImageNet, np. MobileNetV2.
- Zamroz backbone i wytrenuj nowa glowice klasyfikacyjna dla CIFAR-10.
- Odblokuj ostatnie warstwy backbone'u i wykonaj krotki fine-tuning z malym learning rate.
- Porownaj transfer learning z modelem uczonym od zera.

In [ ]:
# Zadanie 4: transfer learning.

# TODO:
# - utworz MobileNetV2 pretrained na ImageNet,
# - przygotuj preprocessing dla CIFAR-10,
# - zamroz features i wymien classifier,
# - wytrenuj nowa glowice,
# - odblokuj wybrane koncowe bloki i wykonaj fine-tuning z malym learning rate,
# - dopisz oba wyniki do torch_rows.

# Preprocessing (Resize do 96x96 + normalizacja ImageNet) jest w transfer_*_transform z Zadania 1a.
transfer_train_loader, transfer_val_loader = make_loaders(transfer_train_transform, transfer_eval_transform)

weights = MobileNet_V2_Weights.IMAGENET1K_V1
transfer_model = models.mobilenet_v2(weights=weights)

# Etap 1: zamrażamy cały backbone (część features) - uczy się tylko nowa głowica.
for param in transfer_model.features.parameters():
    param.requires_grad = False

# Nowa głowica: 1280 cech z backbone'u -> 10 klas CIFAR-10 (oryginalna miała 1000 klas ImageNet).
# Dropout 0.2 jak w oryginalnym MobileNetV2.
transfer_model.classifier = nn.Sequential(
    nn.Dropout(0.2),
    nn.Linear(transfer_model.last_channel, 10),
)
all_params = sum(p.numel() for p in transfer_model.parameters())
print(f"parametry trenowalne: {count_parameters(transfer_model):,d} z {all_params:,d}")

HEAD_EPOCHS = 5
transfer_model, transfer_history, transfer_row = train_torch_model(
    transfer_model,
    "MobileNetV2 - zamrożony backbone",
    transfer_train_loader,
    transfer_val_loader,
    epochs=HEAD_EPOCHS,
    learning_rate=0.001,
    test_transform=transfer_eval_transform,
)
torch_rows.append(transfer_row)

# Etap 2: fine-tuning. features ma 19 modułów (0-18); odblokowujemy ostatnie 4:
# bloki 15-17 (najbardziej "semantyczne" cechy) i końcowy Conv 1x1 (moduł 18).
# Wczesne bloki (krawędzie, tekstury) są uniwersalne, więc zostają zamrożone.
# Mały learning rate (1e-4, 10x mniejszy), żeby nie zniszczyć cech wyuczonych na ImageNet.
FINE_TUNE_FROM = 15
FINE_TUNE_EPOCHS = 8
for param in transfer_model.features[FINE_TUNE_FROM:].parameters():
    param.requires_grad = True
print(f"parametry trenowalne po odblokowaniu: {count_parameters(transfer_model):,d} z {all_params:,d}")

transfer_model, fine_tune_history, fine_tune_row = train_torch_model(
    transfer_model,
    "MobileNetV2 - fine-tuning",
    transfer_train_loader,
    transfer_val_loader,
    epochs=FINE_TUNE_EPOCHS,
    learning_rate=0.0001,
    test_transform=transfer_eval_transform,
)
torch_rows.append(fine_tune_row)

plot_history(transfer_history, "MobileNetV2 - zamrożony backbone")
plot_history(fine_tune_history, "MobileNetV2 - fine-tuning")
print_score_table(torch_rows)

## Wynik koncowy

- Uzyj go jako koncowego raportu: wybierz najlepszy model na podstawie walidacji.
- Dopiero potem policz wynik na zbiorze testowym i wyswietl macierz pomylek.
- Wyniku testowego nie uzywaj do dobierania hiperparametrow.


In [ ]:
# Wynik koncowy.

# TODO:
# - upewnij sie, ze wiersze w torch_rows przechowuja model_object oraz test_transform,
# - wybierz najlepszy model na podstawie validation accuracy,
# - policz accuracy i classification_report na tescie,
# - pokaz macierz pomylek,
# - napisz 2-3 zdania komentarza pod komorka.

test_loader = ...
best_row = ...
best_model = ...

test_pred = ...
test_true = ...

print(...)
print(...)

...
